In [ ]:
!pip install keras_tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 3.7 MB/s eta 0:00:00


In [ ]:
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

Load, prepare the data

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/DPL/Project/Data/balanced_spam.csv')

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

message_train = train_df['message'].values
label_train = train_df['label'].values
message_test = test_df['message'].values
label_test = test_df['label'].values

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((train_df['message'].values, train_df['label'].values))
test_dataset = tf.data.Dataset.from_tensor_slices((test_df['message'].values, test_df['label'].values))
train_dataset.element_spec

(TensorSpec(shape=(), dtype=tf.string, name=None),
 TensorSpec(shape=(), dtype=tf.int64, name=None))

In [ ]:
BUFFER_SIZE = 10000
BATCH_SIZE = 64
train_dataset = train_dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
VOCAB_SIZE = 1000
encoder = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE)
encoder.adapt(train_dataset.map(lambda text, label: text))

Define the model

In [ ]:
def model_builder(hp):
  model = keras.Sequential()

  model.add(encoder)

  model.add(keras.layers.Embedding(
        input_dim = len(encoder.get_vocabulary()),
        output_dim = hp.Int('embedding_output_dim', min_value = 32, max_value = 128, step = 32),
        mask_zero = True))

  model.add(keras.layers.Bidirectional(
      keras.layers.LSTM(hp.Int('lstm_units', min_value = 32, max_value = 128, step = 32))))

  model.add(keras.layers.Dense(
      units=hp.Int('dense_units', min_value = 32, max_value = 128, step = 32), activation = 'relu'))

  model.add(keras.layers.Dense(1, activation='sigmoid'))

  hp_learning_rate = hp.Choice('learning_rate', values = [1e-2, 1e-3, 1e-4])

  model.compile(optimizer=keras.optimizers.Adam(learning_rate = hp_learning_rate),
                loss=keras.losses.BinaryCrossentropy(),
                metrics=['accuracy'])

  return model

Perform hypertuning

In [ ]:
tuner = kt.Hyperband(model_builder,
                     objective = 'val_accuracy',
                     max_epochs = 10,
                     factor = 3,
                     directory = 'my_dir',
                     project_name = 'intro_to_kt')

In [ ]:
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)
tuner.search(message_train, label_train, epochs=15, validation_split=0.2, callbacks=[stop_early])

best_hps = tuner.get_best_hyperparameters(num_trials = 1)[0]
embedding_output_dim = best_hps.get('embedding_output_dim')
lstm_units = best_hps.get('lstm_units')
dense_units = best_hps.get('dense_units')
learning_rate = best_hps.get('learning_rate')

print(f"""
The optimal embedding output dimension: {embedding_output_dim},
The optimal number of LSTM units: {lstm_units},
The optimal number of units in the dense layer: {dense_units},
The optimal learning rate for the optimizer: {learning_rate}.
""")

Trial 30 Complete [00h 01m 01s]
val_accuracy: 0.9976246953010559

Best val_accuracy So Far: 0.9984164834022522
Total elapsed time: 00h 22m 07s

The optimal embedding output dimension: 32,
The optimal number of LSTM units: 128,
The optimal number of units in the dense layer: 32,
The optimal learning rate for the optimizer: 0.01.



Train the model

In [ ]:
best_model = tf.keras.Sequential([
    encoder,
    tf.keras.layers.Embedding(
        input_dim = len(encoder.get_vocabulary()),
        output_dim = embedding_output_dim,
        mask_zero = True
    ),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(lstm_units)),
    tf.keras.layers.Dense(dense_units, activation = 'relu'),
    tf.keras.layers.Dense(1, activation = 'sigmoid')
])

best_model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate = learning_rate),
                    loss = keras.losses.BinaryCrossentropy(),
                    metrics = ['accuracy'])


In [ ]:
history = best_model.fit(train_dataset, epochs=10)

Epoch 1/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 14s 51ms/step - accuracy: 0.9032 - loss: 0.2497
Epoch 2/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 10s 51ms/step - accuracy: 0.9906 - loss: 0.0300
Epoch 3/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 10s 52ms/step - accuracy: 0.9961 - loss: 0.0120
Epoch 4/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 10s 51ms/step - accuracy: 0.9973 - loss: 0.0063
Epoch 5/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 0.9989 - loss: 0.0033
Epoch 6/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 0.9998 - loss: 0.0016
Epoch 7/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 10s 52ms/step - accuracy: 0.9999 - loss: 6.4894e-04
Epoch 8/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 10s 52ms/step - accuracy: 0.9998 - loss: 0.0010
Epoch 9/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 10s 52ms/step - accuracy: 0.9998 - loss: 9.5670e-04
Epoch 10/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 11s 53ms/step - accuracy: 0.9997 - loss: 0.0011


In [ ]:
test_loss, test_acc = best_model.evaluate(test_dataset)
print(f"Test accuracy: {test_acc}")

85/85 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9980 - loss: 0.0082
Test accuracy: 0.9983370304107666


In [ ]:
best_model.save('/content/drive/MyDrive/DPL/Project/Models/best_model.keras')